# ESG Portfolio Optimization - Clean Version

This notebook fixes the logic step by step.

Important idea: we are treating the ESG column as **ESG risk**, because your file is named `ESGrisk.csv`. That means lower ESG risk is better. If your dataset actually means higher ESG score is better, tell me and we will flip this logic.


## Step 1 - Import Libraries

Run this first. These are the only libraries needed for the cleaned version.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

TRADING_DAYS = 252

print("Libraries loaded")


## Step 2 - Set Project Choices

These are the assumptions of the model. We keep them in one place so the project is easier to explain.


In [ ]:
MAX_WEIGHT = 0.30          # no stock can be more than 30% of the portfolio
MAX_ESG_RISK = 22.0        # portfolio ESG risk must be <= this number
RISK_AVERSION = 1.0        # higher means we punish risk more
ESG_PREFERENCE = 0.05      # higher means we reward ESG quality more
NUM_MONTE_CARLO = 100_000
RANDOM_SEED = 42


## Step 3 - Load ESG Data

The original column has a trailing space: `ESG Score `. We strip spaces from all column names so this bug disappears.


In [ ]:
esg_df = pd.read_csv("data/ESGrisk.csv")
esg_df.columns = esg_df.columns.str.strip()
esg_df["Ticker"] = esg_df["Ticker"].str.strip()
esg_df = esg_df.set_index("Ticker")
esg_df = esg_df.rename(columns={"ESG Score": "ESG_Risk"})

esg_df.head()


## Step 4 - Load Close Prices

The price CSV came from yfinance, so it has two header rows. We only need the adjusted `Close` prices.


In [ ]:
prices_raw = pd.read_csv(
    "data/stock_prices_auto_adjusted.csv",
    header=[0, 1],
    index_col=0,
    parse_dates=True
)

close_prices = prices_raw["Close"].copy()
close_prices.columns = close_prices.columns.str.strip()
close_prices = close_prices.sort_index()
close_prices = close_prices.dropna(how="all")

close_prices.head()


## Step 5 - Align Price Data and ESG Data

This is one of the biggest fixes.

The tickers in the price file and the ESG file are not in the same order. If we do not align by ticker, the optimizer gives the wrong ESG score to the wrong stock.


In [ ]:
common_tickers = close_prices.columns.intersection(esg_df.index)

close_prices = close_prices.loc[:, common_tickers]
esg_df = esg_df.loc[common_tickers]

print("Aligned tickers:")
print(list(common_tickers))

print()
print("Same order now?")
print(list(close_prices.columns) == list(esg_df.index))


## Step 6 - Calculate Returns and Risk Inputs

We calculate daily returns, then annualize expected returns and covariance.

Mathematical note: annualizing assumes roughly 252 trading days in a year.


In [ ]:
daily_returns = close_prices.pct_change(fill_method=None).dropna(how="any")

mean_returns = daily_returns.mean() * TRADING_DAYS
cov_matrix = daily_returns.cov() * TRADING_DAYS

print("Number of daily return observations:", len(daily_returns))
print("Annualized expected returns:")
print(mean_returns.round(4))


## Step 7 - Convert ESG Risk into ESG Quality

Because we are treating ESG as risk, lower is better.

To make it usable in the objective function, we convert it into a 0-1 quality score where higher is better.


In [ ]:
esg_risk = esg_df["ESG_Risk"]

esg_quality = 1 - ((esg_risk - esg_risk.min()) / (esg_risk.max() - esg_risk.min()))
esg_quality.name = "ESG_Quality"

pd.DataFrame({
    "ESG_Risk": esg_risk,
    "ESG_Quality": esg_quality
}).sort_values("ESG_Risk")


## Step 8 - Define Portfolio Metrics

This function calculates return, risk, ESG risk, and ESG quality for any portfolio weights.

Important: risk is standard deviation, not variance.


In [ ]:
def portfolio_metrics(weights):
    portfolio_return = np.dot(weights, mean_returns)
    portfolio_risk = np.sqrt(weights.T @ cov_matrix.values @ weights)
    portfolio_esg_risk = np.dot(weights, esg_risk)
    portfolio_esg_quality = np.dot(weights, esg_quality)
    
    return {
        "Return": portfolio_return,
        "Risk": portfolio_risk,
        "ESG_Risk": portfolio_esg_risk,
        "ESG_Quality": portfolio_esg_quality,
    }


def portfolio_utility(metrics):
    return (
        metrics["Return"]
        - RISK_AVERSION * metrics["Risk"]
        + ESG_PREFERENCE * metrics["ESG_Quality"]
    )


## Step 9 - Optimize with SLSQP

SLSQP searches directly for the best weights.

Constraints:
- weights sum to 1
- no short selling
- each stock has max weight 30%
- portfolio ESG risk must be less than or equal to 22


In [ ]:
num_assets = len(mean_returns)
initial_weights = np.ones(num_assets) / num_assets
bounds = [(0, MAX_WEIGHT) for _ in range(num_assets)]

constraints = [
    {"type": "eq", "fun": lambda weights: np.sum(weights) - 1},
    {"type": "ineq", "fun": lambda weights: MAX_ESG_RISK - np.dot(weights, esg_risk)},
]


def objective(weights):
    metrics = portfolio_metrics(weights)
    return -portfolio_utility(metrics)

result = minimize(
    objective,
    initial_weights,
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
    options={"maxiter": 1000, "ftol": 1e-12},
)

print("Optimization success:", result.success)
print("Message:", result.message)


## Step 10 - Show the Optimal Portfolio

Only trust these weights if `Optimization success` was `True` in the previous cell.


In [ ]:
optimal_weights = pd.Series(result.x, index=mean_returns.index, name="Weight")
optimal_metrics = portfolio_metrics(result.x)
optimal_metrics["Utility"] = portfolio_utility(optimal_metrics)

print("Optimal metrics:")
print(pd.Series(optimal_metrics).round(4))

print()
print("Optimal weights:")
print(optimal_weights[optimal_weights > 0.001].sort_values(ascending=False).round(4))


## Step 11 - Generate Monte Carlo Portfolios

Monte Carlo is not the true optimizer. It is a random search.

We use it to compare against SLSQP and to visualize the portfolio space. It must use the same constraints as SLSQP.


In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
records = []

for _ in range(NUM_MONTE_CARLO):
    weights = rng.random(num_assets)
    weights = weights / weights.sum()
    
    if weights.max() > MAX_WEIGHT:
        continue
    
    metrics = portfolio_metrics(weights)
    
    if metrics["ESG_Risk"] > MAX_ESG_RISK:
        continue
    
    metrics["Utility"] = portfolio_utility(metrics)
    records.append(metrics)

results_df = pd.DataFrame(records)

print("Valid Monte Carlo portfolios:", len(results_df))
results_df.head()


## Step 12 - Compare SLSQP to Best Monte Carlo Portfolio

SLSQP should usually be better than the best Monte Carlo portfolio because SLSQP searches continuously, while Monte Carlo only checks random portfolios.


In [ ]:
best_mc = results_df.loc[results_df["Utility"].idxmax()]

comparison = pd.DataFrame({
    "SLSQP": pd.Series(optimal_metrics),
    "Best Monte Carlo": best_mc[["Return", "Risk", "ESG_Risk", "ESG_Quality", "Utility"]]
})

comparison.round(4)


## Step 13 - Plot Monte Carlo Cloud and SLSQP Portfolio

This plot checks whether the optimized portfolio makes sense relative to the simulated portfolio space.


In [ ]:
plt.figure(figsize=(10, 6))

scatter = plt.scatter(
    results_df["Risk"],
    results_df["Return"],
    c=results_df["ESG_Risk"],
    cmap="viridis_r",
    alpha=0.45,
    label="Monte Carlo portfolios"
)

plt.colorbar(scatter, label="ESG Risk")

plt.scatter(
    optimal_metrics["Risk"],
    optimal_metrics["Return"],
    color="red",
    marker="*",
    s=300,
    label="SLSQP optimal portfolio"
)

plt.xlabel("Annualized Risk / Volatility")
plt.ylabel("Annualized Expected Return")
plt.title("SLSQP Optimized Portfolio vs Monte Carlo Portfolios")
plt.legend()
plt.grid(True)
plt.show()


## Where the Mathematics Can Go Deeper Later

Once this cleaned version works, the main mathematical improvement is the objective function.

Right now we optimize:

`Return - risk_aversion * Risk + esg_preference * ESG_Quality`

For a stronger final project, we can later replace this with one of these:

1. Maximize Sharpe ratio with ESG constraint.
2. Minimize risk for a target return and target ESG risk.
3. Build an efficient frontier under ESG constraints.
4. Compare normal Markowitz optimization vs ESG-constrained optimization.
